# H1 - Query Expansion Research Pipeline

This notebook studies the query expansion step in isolation before it is connected to retrieval. The goal is not to optimize Recall@K, MRR, nDCG, or any corpus-dependent metric yet. Instead, this notebook checks whether an LLM can generate English, search-ready, faithful, diverse, and self-contained expanded queries from Vietnamese user queries.

Main workflow:

1. Initialize imports, environment variables, and experiment paths.
2. Build synthetic evaluation cases with required concepts, forbidden drift terms, and reference expansions.
3. Define intrinsic query-expansion metrics.
4. Configure models, prompt variants, output parsing, and retry behavior.
5. Run local model/prompt/hyperparameter grids and save comparison tables.
6. Optionally publish the same evaluation to LangSmith for trace inspection and experiment comparison.

Detailed experiment notes are in `h1_experiment.md`.


## 0. Dependencies

This cell lists optional package installs for a fresh notebook environment. The commands are commented out by default so the notebook does not install dependencies unexpectedly. Uncomment only the packages that are missing in your current kernel.


In [ ]:
# Optional first-run installs. Uncomment only if your notebook environment is missing packages.
# %pip install -U langchain langchain-core langchain-groq langchain-openai langsmith python-dotenv pydantic pandas
# Optional semantic metrics. This may download an embedding model the first time it runs.
# %pip install -U sentence-transformers

## 1. Environment And Paths

This cell imports libraries, loads `.env` from the repository root, and creates stable paths for data and experiment outputs. The `find_repo_root` helper makes the notebook usable whether it is launched from the repository root or from another working directory.


In [1]:
from __future__ import annotations

import json
import math
import os
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from itertools import combinations, product
from pathlib import Path
from typing import Any, Iterable, Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

try:
    from langchain_openai import ChatOpenAI
except ImportError:
    ChatOpenAI = None

try:
    from langsmith import Client, traceable
except ImportError:
    Client = None
    traceable = None



def find_repo_root(start: Path) -> Path:
    """Find the repository root so the notebook works from Jupyter or CLI execution."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "agent").exists() and (candidate / "scripts").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env", override=False)
NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "agent"
QUERY_DIR = REPO_ROOT / "scripts" / "query-p1-groupA"
EXPERIMENT_DIR = NOTEBOOK_DIR / "experiments" / "h1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("QUERY_DIR:", QUERY_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)


C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT: D:\University\Projects\Individual projects\Multimodal-Retrieval
QUERY_DIR: D:\University\Projects\Individual projects\Multimodal-Retrieval\scripts\query-p1-groupA
EXPERIMENT_DIR: D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1


## 2. Synthetic Evaluation Data

This cell defines the H1 synthetic dataset. Each case acts like a small unit test for query expansion: it has a Vietnamese input query, required concepts that should be preserved, forbidden terms that indicate drift, and reference expansions for lightweight overlap checks.

The generated JSONL file is saved to `notebooks/agent/experiments/h1/synthetic_cases.jsonl` so the same cases can be reused by local runs and LangSmith evaluation.


In [3]:
# Default number of expanded queries expected from each model call.
DEFAULT_K = 5

# Each synthetic case encodes semantic anchors and drift traps for expand-only evaluation.
SYNTHETIC_CASES: list[dict[str, Any]] = [
    {
        "case_id": "kis_yellow_raincoat_motorbike_night",
        "query_type": "KIS",
        "query": "Tìm cảnh môt người đàn ông mặc áo mưa màu vàng chạy xe máy qua con đường ngập nước vào ban đêm.",
        "required_concepts": {
            "person": ["man", "male person", "rider"],
            "clothing": ["yellow raincoat", "yellow poncho", "yellow waterproof coat"],
            "vehicle": ["motorbike", "motorcycle", "scooter"],
            "scene": ["flooded street", "waterlogged road", "street flooding"],
            "time": ["night", "nighttime", "dark"],
        },
        "forbidden_terms": ["taxi", "car", "daytime", "snow", "woman"],
        "reference_expansions": [
            "man in a yellow raincoat riding a motorcycle through a flooded street at night",
            "nighttime scene of a scooter crossing a waterlogged road with a rider wearing yellow rain gear",
            "male motorbike rider in a yellow poncho on a dark flooded urban road",
            "video frame showing motorcycle rider wearing yellow waterproof coat during street flooding at night",
            "yellow raincoat motorcycle flooded road night search query",
        ],
    },
    {
        "case_id": "kis_red_bus_big_ben",
        "query_type": "KIS",
        "query": "Tìm hình ảnh xe buýt hai tầng màu đỏ chạy ngang qua tháp đồng hồ Big Ben ở London.",
        "required_concepts": {
            "vehicle": ["red double-decker bus", "red double decker bus", "two-level red bus"],
            "landmark": ["Big Ben", "clock tower", "Elizabeth Tower"],
            "location": ["London"],
            "motion": ["passing", "driving past", "moving by"],
        },
        "forbidden_terms": ["Eiffel", "Paris", "yellow bus", "train", "Tokyo"],
        "reference_expansions": [
            "red double-decker bus driving past Big Ben in London",
            "London street scene with a red two-level bus passing the Big Ben clock tower",
            "red bus near Elizabeth Tower landmark in London",
            "video frame of a double decker bus moving by Big Ben",
            "Big Ben London red double-decker bus search query",
        ],
    },
    {
        "case_id": "qa_child_blue_robot_beach",
        "query_type": "QA",
        "query": "Trong đoạn video, đứa trẻ đang cầm đồ chơi robot màu gì trên bãi biển?",
        "required_concepts": {
            "subject": ["child", "kid", "young child"],
            "object": ["toy robot", "robot toy"],
            "color": ["blue"],
            "place": ["beach", "seaside", "sand"],
            "qa_intent": ["what color", "color of the toy robot"],
        },
        "forbidden_terms": ["red", "green", "park", "adult", "kite"],
        "reference_expansions": [
            "child holding a blue toy robot on the beach",
            "what color is the toy robot held by the child on the beach",
            "kid on sand with blue robot toy video frame",
            "beach scene child carrying blue toy robot",
            "answer evidence for blue robot toy held by child at seaside",
        ],
    },
    {
        "case_id": "trake_cooking_egg_pancake",
        "query_type": "TRAKE",
        "query": "Tìm chuỗi cảnh đầu bếp đập trứng vào tô, khuấy bột, rồi đổ bột lên chảo làm bánh pancake.",
        "required_concepts": {
            "actor": ["chef", "cook"],
            "step_1": ["cracking eggs", "breaks eggs", "egg into a bowl"],
            "step_2": ["mixing batter", "whisking batter", "stirring batter"],
            "step_3": ["pouring batter", "batter into a pan", "pancake pan"],
            "dish": ["pancake", "pancakes"],
        },
        "forbidden_terms": ["omelet", "grill", "microwave", "cake frosting"],
        "reference_expansions": [
            "chef cracks eggs into a bowl then mixes batter and pours it into a pan for pancakes",
            "sequence of cooking steps cracking eggs whisking pancake batter pouring batter on pan",
            "cook preparing pancakes by breaking eggs stirring batter and pouring into skillet",
            "temporal video query chef egg bowl batter pan pancake",
            "pancake preparation sequence from eggs to batter poured into frying pan",
        ],
    },
    {
        "case_id": "kis_woman_ao_dai_lotus",
        "query_type": "KIS",
        "query": "Tìm cảnh người phụ nữ mặc áo dài trắng đứng bên hồ sen và cầm nón lá.",
        "required_concepts": {
            "subject": ["woman", "female person"],
            "clothing": ["white ao dai", "white Vietnamese dress", "white traditional dress"],
            "place": ["lotus pond", "lotus lake"],
            "object": ["conical hat", "non la", "Vietnamese conical hat"],
        },
        "forbidden_terms": ["kimono", "red dress", "umbrella", "beach", "man"],
        "reference_expansions": [
            "woman wearing a white ao dai standing beside a lotus pond holding a conical hat",
            "Vietnamese woman in white traditional dress near lotus lake with non la",
            "female person white ao dai lotus pond Vietnamese conical hat",
            "search query for woman in white Vietnamese dress beside lotus flowers holding hat",
            "video frame white ao dai woman lotus pond conical hat",
        ],
    },
    {
        "case_id": "qa_basketball_scoreboard_final",
        "query_type": "QA",
        "query": "Bảng điểm trong trận bóng rổ hiển thị tỉ số cuối cùng là bao nhiêu?",
        "required_concepts": {
            "object": ["scoreboard", "score board"],
            "sport": ["basketball"],
            "intent": ["final score", "score displayed", "what score"],
            "evidence": ["numbers", "digits"],
        },
        "forbidden_terms": ["football", "soccer", "tennis", "baseball", "halftime"],
        "reference_expansions": [
            "basketball scoreboard showing the final score numbers",
            "what final score is displayed on the basketball scoreboard",
            "video frame evidence of basketball score board digits at end of game",
            "basketball game final score visible on scoreboard",
            "read the numbers on the basketball scoreboard final result",
        ],
    },
    {
        "case_id": "trake_delivery_doorbell_package",
        "query_type": "TRAKE",
        "query": "Tìm chuỗi cảnh người giao hàng đặt gói hàng trước cửa, bấm chuông, rồi rời đi.",
        "required_concepts": {
            "actor": ["delivery person", "courier", "delivery worker"],
            "object": ["package", "parcel", "box"],
            "step_1": ["places package", "sets parcel", "leaves box"],
            "step_2": ["rings doorbell", "presses doorbell", "door bell"],
            "step_3": ["walks away", "leaves", "departing"],
        },
        "forbidden_terms": ["mailbox", "garage", "steals", "returns", "police"],
        "reference_expansions": [
            "delivery person places a package at the front door rings the doorbell and walks away",
            "courier leaves parcel by doorway presses door bell then departs",
            "temporal query package delivery doorbell courier walking away",
            "front door security camera sequence delivery worker sets box rings bell leaves",
            "parcel drop off followed by doorbell press and courier leaving",
        ],
    },
    {
        "case_id": "kis_rocket_countdown_launchpad",
        "query_type": "KIS",
        "query": "Tìm khoảnh khắc tên lửa trên bệ phóng có khói trắng bốc lên trong lúc đếm ngược phóng.",
        "required_concepts": {
            "object": ["rocket", "space rocket"],
            "place": ["launch pad", "launchpad"],
            "visual": ["white smoke", "steam", "smoke plume"],
            "event": ["countdown", "launch countdown"],
        },
        "forbidden_terms": ["airplane", "fireworks", "missile strike", "landing", "rain"],
        "reference_expansions": [
            "rocket on a launch pad with white smoke during launch countdown",
            "space rocket countdown scene with steam rising from the launchpad",
            "video frame rocket launch pad white smoke plume before liftoff",
            "launch countdown showing rocket surrounded by white vapor",
            "rocket launchpad smoke countdown search query",
        ],
    },
]


def write_jsonl(rows: Iterable[dict[str, Any]], path: Path) -> None:
    """Persist experiment fixtures or predictions as UTF-8 JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


SYNTHETIC_DATA_PATH = EXPERIMENT_DIR / "synthetic_cases.jsonl"
write_jsonl(SYNTHETIC_CASES, SYNTHETIC_DATA_PATH)
print(f"Saved {len(SYNTHETIC_CASES)} synthetic cases to {SYNTHETIC_DATA_PATH}")


Saved 8 synthetic cases to D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1\synthetic_cases.jsonl


## 3. Intrinsic Metric Implementation

This cell implements metrics that do not require a retrieval corpus. The first helpers normalize text and compute lexical overlap. The middle helpers measure concept coverage, hallucination avoidance, query diversity, English readiness, and length quality. The final function combines these into one metric dictionary for each model output.

`overall_score` is only a ranking helper for H1 experiments. It should not be interpreted as retrieval quality.


In [5]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "at", "by", "for", "with", "from",
    "into", "during", "showing", "scene", "video", "frame", "query", "search", "what", "is", "are",
}
VIETNAMESE_DIACRITICS = set("aăâeêioôơuưyAĂÂEÊIOÔƠUƯYáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵđĐ")


def normalize_text(text: str) -> str:
    """Lowercase and remove punctuation for lightweight lexical matching."""
    text = text.casefold()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str, keep_stopwords: bool = False) -> list[str]:
    """Tokenize normalized English text, optionally preserving stopwords for n-grams."""
    tokens = re.findall(r"[a-z0-9]+", normalize_text(text))
    if keep_stopwords:
        return tokens
    return [token for token in tokens if token not in STOPWORDS]


def ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    """Return contiguous n-grams used by distinct-n diversity metrics."""
    if len(tokens) < n:
        return []
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


def deduplicate_queries(queries: Iterable[str]) -> list[str]:
    """Normalize whitespace and remove duplicate queries while preserving order."""
    result: list[str] = []
    seen: set[str] = set()
    for item in queries:
        query = " ".join(str(item).strip().split())
        key = query.casefold()
        if query and key not in seen:
            result.append(query)
            seen.add(key)
    return result


def jaccard(a: set[str], b: set[str]) -> float:
    """Compute Jaccard similarity with sensible behavior for empty sets."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def concept_coverage(expanded_queries: list[str], required_concepts: dict[str, list[str]]) -> tuple[float, dict[str, bool]]:
    """Measure how many required semantic concepts appear in the generated queries."""
    joined = normalize_text(" ".join(expanded_queries))
    coverage: dict[str, bool] = {}
    for concept, aliases in required_concepts.items():
        coverage[concept] = any(normalize_text(alias) in joined for alias in aliases)
    if not coverage:
        return 1.0, coverage
    return sum(coverage.values()) / len(coverage), coverage


def forbidden_avoidance(expanded_queries: list[str], forbidden_terms: list[str]) -> tuple[float, list[str]]:
    """Penalize generated queries that introduce known drift/hallucination terms."""
    joined = normalize_text(" ".join(expanded_queries))
    hits = [term for term in forbidden_terms if normalize_text(term) in joined]
    if not forbidden_terms:
        return 1.0, []
    return 1.0 - (len(hits) / len(forbidden_terms)), hits


def reference_token_overlap(expanded_queries: list[str], reference_expansions: list[str]) -> float:
    """Compare generated vocabulary against human-written reference expansions."""
    generated = set(tokenize(" ".join(expanded_queries)))
    reference = set(tokenize(" ".join(reference_expansions)))
    return jaccard(generated, reference)


def pairwise_jaccard_diversity(expanded_queries: list[str]) -> float:
    """Estimate lexical diversity as one minus average pairwise Jaccard similarity."""
    pairs = list(combinations(expanded_queries, 2))
    if not pairs:
        return 0.0
    similarities = []
    for left, right in pairs:
        similarities.append(jaccard(set(tokenize(left)), set(tokenize(right))))
    return 1.0 - (sum(similarities) / len(similarities))


def distinct_n(expanded_queries: list[str], n: int) -> float:
    """Compute distinct-n over all expanded queries as a simple diversity score."""
    all_ngrams: list[tuple[str, ...]] = []
    for query in expanded_queries:
        all_ngrams.extend(ngrams(tokenize(query, keep_stopwords=True), n))
    if not all_ngrams:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)


def englishish_score(expanded_queries: list[str]) -> float:
    """Approximate whether outputs are English search strings rather than Vietnamese text."""
    if not expanded_queries:
        return 0.0
    scores: list[float] = []
    for query in expanded_queries:
        chars = [c for c in query if not c.isspace()]
        if not chars:
            scores.append(0.0)
            continue
        ascii_ratio = sum(ord(c) < 128 for c in chars) / len(chars)
        vietnamese_ratio = sum(c in VIETNAMESE_DIACRITICS and ord(c) >= 128 for c in chars) / len(chars)
        scores.append(max(0.0, min(1.0, ascii_ratio - vietnamese_ratio)))
    return sum(scores) / len(scores)


def length_score(expanded_queries: list[str], min_words: int = 5, max_words: int = 18) -> float:
    """Reward query lengths that are useful for search without becoming verbose answers."""
    if not expanded_queries:
        return 0.0
    per_query: list[float] = []
    for query in expanded_queries:
        word_count = len(tokenize(query, keep_stopwords=True))
        if min_words <= word_count <= max_words:
            per_query.append(1.0)
        elif word_count < min_words:
            per_query.append(max(0.0, word_count / min_words))
        else:
            per_query.append(max(0.0, 1.0 - ((word_count - max_words) / max_words)))
    return sum(per_query) / len(per_query)


_EMBEDDER = None


def get_sentence_embedder(model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    """Lazy-load the optional sentence-transformer used for semantic metrics."""
    global _EMBEDDER
    if _EMBEDDER is None:
        from sentence_transformers import SentenceTransformer

        _EMBEDDER = SentenceTransformer(model_name)
    return _EMBEDDER


def cosine(left: Iterable[float], right: Iterable[float]) -> float:
    """Compute cosine similarity without adding a heavy numerical dependency."""
    left_list = list(float(x) for x in left)
    right_list = list(float(x) for x in right)
    numerator = sum(a * b for a, b in zip(left_list, right_list))
    left_norm = math.sqrt(sum(a * a for a in left_list))
    right_norm = math.sqrt(sum(b * b for b in right_list))
    if left_norm == 0.0 or right_norm == 0.0:
        return 0.0
    return numerator / (left_norm * right_norm)


def optional_embedding_scores(expanded_queries: list[str], reference_expansions: list[str], enabled: bool = False) -> dict[str, float | None]:
    """Return optional semantic relevance/diversity scores when embeddings are enabled."""
    if not enabled:
        return {"embedding_relevance": None, "embedding_diversity": None}
    try:
        embedder = get_sentence_embedder()
        candidate_embeddings = embedder.encode(expanded_queries, normalize_embeddings=True)
        reference_embeddings = embedder.encode(reference_expansions, normalize_embeddings=True)

        relevance_scores: list[float] = []
        for candidate in candidate_embeddings:
            relevance_scores.append(max(cosine(candidate, ref) for ref in reference_embeddings))

        diversity_scores: list[float] = []
        for i, j in combinations(range(len(candidate_embeddings)), 2):
            diversity_scores.append(1.0 - cosine(candidate_embeddings[i], candidate_embeddings[j]))

        return {
            "embedding_relevance": float(sum(relevance_scores) / len(relevance_scores)) if relevance_scores else None,
            "embedding_diversity": float(sum(diversity_scores) / len(diversity_scores)) if diversity_scores else None,
        }
    except Exception as exc:
        print(f"Embedding metrics skipped: {exc}")
        return {"embedding_relevance": None, "embedding_diversity": None}


def compute_query_expansion_metrics(
    expanded_queries: list[str],
    case: dict[str, Any],
    k: int = DEFAULT_K,
    use_embeddings: bool = False,
) -> dict[str, Any]:
    """Compute all intrinsic H1 metrics for one generated query-expansion output."""
    expanded_queries = deduplicate_queries(expanded_queries)
    required_score, concept_hits = concept_coverage(expanded_queries, case["required_concepts"])
    forbidden_score, forbidden_hits = forbidden_avoidance(expanded_queries, case["forbidden_terms"])
    embedding = optional_embedding_scores(expanded_queries, case["reference_expansions"], enabled=use_embeddings)

    metrics: dict[str, Any] = {
        "exact_k": 1.0 if len(expanded_queries) == k else 0.0,
        "unique_ratio": len(set(q.casefold() for q in expanded_queries)) / max(1, len(expanded_queries)),
        "required_concept_coverage": required_score,
        "forbidden_avoidance": forbidden_score,
        "reference_token_overlap": reference_token_overlap(expanded_queries, case["reference_expansions"]),
        "pairwise_lexical_diversity": pairwise_jaccard_diversity(expanded_queries),
        "distinct_1": distinct_n(expanded_queries, 1),
        "distinct_2": distinct_n(expanded_queries, 2),
        "englishish_score": englishish_score(expanded_queries),
        "length_score": length_score(expanded_queries),
        "concept_hits": concept_hits,
        "forbidden_hits": forbidden_hits,
        **embedding,
    }

    # Weighted score is a ranking helper, not an absolute measure of retrieval quality.
    base_weights = {
        "exact_k": 0.12,
        "unique_ratio": 0.10,
        "required_concept_coverage": 0.24,
        "forbidden_avoidance": 0.16,
        "reference_token_overlap": 0.10,
        "pairwise_lexical_diversity": 0.10,
        "distinct_2": 0.08,
        "englishish_score": 0.05,
        "length_score": 0.05,
    }
    weighted_sum = sum(metrics[name] * weight for name, weight in base_weights.items())
    metrics["overall_score"] = weighted_sum / sum(base_weights.values())
    metrics["query_drift_risk"] = 1.0 - ((metrics["required_concept_coverage"] + metrics["forbidden_avoidance"]) / 2.0)
    return metrics


In [68]:
smoke_case = SYNTHETIC_CASES[0]
compute_query_expansion_metrics(smoke_case["reference_expansions"], smoke_case)

{'exact_k': 1.0,
 'unique_ratio': 1.0,
 'required_concept_coverage': 1.0,
 'forbidden_avoidance': 1.0,
 'reference_token_overlap': 1.0,
 'pairwise_lexical_diversity': 0.7890873015873016,
 'distinct_1': 0.6,
 'distinct_2': 0.9,
 'englishish_score': 1.0,
 'length_score': 1.0,
 'concept_hits': {'person': True,
  'clothing': True,
  'vehicle': True,
  'scene': True,
  'time': True},
 'forbidden_hits': [],
 'embedding_relevance': None,
 'embedding_diversity': None,
 'overall_score': 0.9709087301587301,
 'query_drift_risk': 0.0}

## 4. Models, Prompts, And Output Parsing

This cell contains the core query-expansion runtime. It defines provider-agnostic model configs, generation hyperparameters, prompt variants, a Pydantic output schema, JSON parsing utilities, and a retry loop that tries to collect exactly `k` unique expanded queries.


In [65]:
load_dotenv(REPO_ROOT / ".env", override=False)
print(os.getenv("GROQ_API_KEY"))
print(os.getenv("LANGSMITH_API_KEY"))

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
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


In [45]:
@dataclass(frozen=True)
class ModelConfig:
    """Provider-agnostic model configuration used by the experiment grid."""

    alias: str
    provider: Literal["groq", "openai_compatible"]
    model_name: str
    api_key_env: str
    base_url_env: str | None = None
    supports_reasoning_effort: bool = False
    notes: str = ""


@dataclass(frozen=True)
class GenerationParams:
    """Generation hyperparameters that can be swept in H1 experiments."""

    temperature: float = 0.4
    max_tokens: int = 2048
    timeout: int = 60
    max_retries: int = 2
    reasoning_effort: Literal["low", "medium", "high"] | None = "medium"
    output_mode: Literal["json_prompt", "structured"] = "json_prompt"


MODEL_CONFIGS: dict[str, ModelConfig] = {
    "gpt_oss_120b": ModelConfig(
        alias="gpt_oss_120b",
        provider="groq",
        # model_name=os.getenv("GPT_OS_MODEL", os.getenv("GPT_OSS_MODEL", "openai/gpt-oss-120b")),
        model_name="openai/gpt-oss-120b",
        api_key_env="GROQ_API_KEY",
        supports_reasoning_effort=True,
        notes="Uses the GPT-OSS/GPT-OS style model from the current notebook. Override with GPT_OS_MODEL or GPT_OSS_MODEL.",
    ),
    # "gemma": ModelConfig(
    #     alias="gemma",
    #     provider="groq",
    #     model_name=os.getenv("GEMMA_MODEL", "gemma2-9b-it"),
    #     api_key_env="GROQ_API_KEY",
    #     notes="Gemma via Groq by default. Override with GEMMA_MODEL if your provider uses another ID.",
    # ),
    # "nemotron": ModelConfig(
    #     alias="nemotron",
    #     provider="openai_compatible",
    #     model_name=os.getenv("NEMOTRON_MODEL", "nvidia/llama-3.1-nemotron-ultra-253b-v1"),
    #     api_key_env="NEMOTRON_API_KEY",
    #     base_url_env="NEMOTRON_BASE_URL",
    #     notes="Set NEMOTRON_BASE_URL and NEMOTRON_API_KEY for OpenRouter, NVIDIA NIM, or a local OpenAI-compatible endpoint.",
    # ),
}

PROMPT_VARIANTS: dict[str, str] = {
    "strict_multimedia_v1": '''
You are a query-expansion planner for a large-scale multimedia retrieval system.

The input query may be written in Vietnamese. Generate search-ready queries in English.

Requirements:
- Produce exactly {k} unique expanded queries.
- Preserve every reliable fact from the original query.
- Do not invent names, dates, organizations, people, locations, colors, scores, or events that are not explicitly present.
- Make every query self-contained so it can be searched independently.
- Diversify retrieval intent across visual evidence, OCR text, ASR/subtitles, event/action description, entities/relations, and concise keyword matching.
- Use natural English, not literal word-for-word translation.
- Do not answer the query.
- Return only valid JSON with this schema: {{"queries": ["query 1", "query 2"]}}.
'''.strip(),
    "keyword_control_v1": '''
You optimize Vietnamese-to-English query expansion for multimedia search.

Generate exactly {k} English queries. Each query must keep the original meaning but use a different search angle.

Good expansion types:
- visual objects, people, clothing, colors, scene, camera-visible actions
- OCR or text visible on screen when relevant
- ASR/subtitle/narration wording when relevant
- short keyword-style query for lexical search
- temporal sequence query for TRAKE-style prompts

Hard constraints:
- No new entities or unsupported details.
- No answers, explanations, bullet points, Markdown, or prose outside JSON.
- Every query must be independently understandable.
- Return JSON only: {{"queries": ["..."]}}.
'''.strip(),
    "minimal_translation_v1": '''
Translate and expand the user query into exactly {k} concise English search queries for multimedia retrieval.
Keep the facts unchanged, avoid hallucination, vary wording, and return only JSON: {{"queries": ["..."]}}.
'''.strip(),
}


In [67]:
class QueryExpansionOutput(BaseModel):
    """Validated schema expected from every query-expansion model call."""

    queries: list[str] = Field(description="Unique, self-contained English search queries.")

    @field_validator("queries")
    @classmethod
    def clean_queries(cls, values: list[str]) -> list[str]:
        """Clean model output and fail fast if no usable query remains."""
        cleaned = deduplicate_queries(values)
        if not cleaned:
            raise ValueError("The model returned no usable queries.")
        return cleaned


def model_is_ready(config: ModelConfig) -> tuple[bool, str]:
    """Check whether required provider packages and environment variables are available."""
    if not os.getenv(config.api_key_env):
        return False, f"Missing {config.api_key_env}"
    if config.base_url_env and not os.getenv(config.base_url_env):
        return False, f"Missing {config.base_url_env}"
    if config.provider == "openai_compatible" and ChatOpenAI is None:
        return False, "Missing langchain-openai package"
    return True, "ready"


_LLM_CACHE: dict[tuple[Any, ...], Any] = {}


def make_llm(config: ModelConfig, params: GenerationParams):
    """Instantiate and cache a LangChain chat model for a model/hyperparameter pair."""
    ready, reason = model_is_ready(config)
    if not ready:
        raise RuntimeError(f"Model {config.alias} is not ready: {reason}")

    cache_key = (config.alias, config.provider, config.model_name, params.temperature, params.max_tokens, params.timeout, params.reasoning_effort)
    if cache_key in _LLM_CACHE:
        return _LLM_CACHE[cache_key]

    if config.provider == "groq":
        kwargs: dict[str, Any] = {
            "model": config.model_name,
            "temperature": params.temperature,
            "max_tokens": params.max_tokens,
            "timeout": params.timeout,
            "max_retries": params.max_retries,
        }
        if config.supports_reasoning_effort and params.reasoning_effort:
            kwargs["reasoning_effort"] = params.reasoning_effort
        llm = ChatGroq(**kwargs)
    elif config.provider == "openai_compatible":
        if ChatOpenAI is None:
            raise RuntimeError("Install langchain-openai to use OpenAI-compatible providers.")
        llm = ChatOpenAI(
            model=config.model_name,
            base_url=os.getenv(config.base_url_env or "OPENAI_BASE_URL"),
            api_key=os.getenv(config.api_key_env),
            temperature=params.temperature,
            max_tokens=params.max_tokens,
            timeout=params.timeout,
            max_retries=params.max_retries,
        )
    else:
        raise ValueError(f"Unsupported provider: {config.provider}")

    _LLM_CACHE[cache_key] = llm
    return llm


def extract_json_object(text: str) -> dict[str, Any]:
    """Extract the first JSON object from model text, tolerating fenced JSON blocks."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text.strip()).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def parse_query_expansion_response(response: Any) -> QueryExpansionOutput:
    """Convert structured or raw LLM responses into QueryExpansionOutput."""
    if isinstance(response, QueryExpansionOutput):
        return response
    if isinstance(response, dict):
        return QueryExpansionOutput.model_validate(response)
    if isinstance(response, BaseMessage):
        content = response.content
    else:
        content = getattr(response, "content", response)
    if isinstance(content, list):
        content = "\n".join(str(item) for item in content)
    data = extract_json_object(str(content))
    return QueryExpansionOutput.model_validate(data)


def build_prompt(prompt_id: str) -> ChatPromptTemplate:
    """Build the system/human prompt pair for the selected prompt variant."""
    system_prompt = PROMPT_VARIANTS[prompt_id]
    return ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            (
                "human",
                '''Original query:\n{query}\n\nNumber of expanded queries required: {k}\n'''.strip(),
            ),
        ]
    )


def invoke_expander_once(
    query: str,
    k: int,
    model_config: ModelConfig,
    prompt_id: str,
    params: GenerationParams,
) -> tuple[QueryExpansionOutput, dict[str, Any]]:
    """Call one model once and return parsed output plus latency/usage metadata."""
    llm = make_llm(model_config, params)
    prompt = build_prompt(prompt_id)
    payload = {"query": query, "k": k}
    start = time.perf_counter()

    if params.output_mode == "structured":
        try:
            structured_llm = llm.with_structured_output(QueryExpansionOutput, method="json_schema", strict=True)
            response = (prompt | structured_llm).invoke(payload)
        except Exception:
            response = (prompt | llm).invoke(payload)
    else:
        response = (prompt | llm).invoke(payload)

    elapsed = time.perf_counter() - start
    usage = getattr(response, "usage_metadata", None)
    parsed = parse_query_expansion_response(response)
    return parsed, {"latency_sec": elapsed, "usage_metadata": usage}


def expand_query(
    query: str,
    k: int,
    model_config: ModelConfig,
    prompt_id: str,
    params: GenerationParams,
    max_attempts: int = 3,
) -> dict[str, Any]:
    """Generate exactly k unique expanded queries, retrying to fill missing items."""
    if not query.strip():
        raise ValueError("Query must not be empty.")
    if k < 1 or k > 20:
        raise ValueError("k must be between 1 and 20.")

    collected: list[str] = []
    usage_events: list[dict[str, Any]] = []
    total_latency = 0.0
    current_query = query.strip()

    for attempt in range(1, max_attempts + 1):
        result, run_info = invoke_expander_once(current_query, k, model_config, prompt_id, params)
        usage_events.append(run_info)
        total_latency += float(run_info["latency_sec"])
        collected = deduplicate_queries([*collected, *result.queries])
        if len(collected) >= k:
            return {
                "expanded_queries": collected[:k],
                "attempts": attempt,
                "latency_sec": total_latency,
                "usage_events": usage_events,
            }

        # Ask only for missing items on retry so duplicate-heavy models can recover.
        missing = k - len(collected)
        current_query = (
            f"{query.strip()}\n\n"
            "Already generated queries that must not be repeated:\n"
            + "\n".join(f"- {item}" for item in collected)
            + f"\nGenerate {missing} additional distinct English search queries."
        )

    return {
        "expanded_queries": collected[:k],
        "attempts": max_attempts,
        "latency_sec": total_latency,
        "usage_events": usage_events,
    }


pd.DataFrame([asdict(config) for config in MODEL_CONFIGS.values()])


,alias,provider,model_name,api_key_env,base_url_env,supports_reasoning_effort,notes
0,gpt_oss_120b,groq,openai/gpt-oss-120b,GROQ_API_KEY,None,True,Uses the GPT-OSS/GPT-OS style model from the c...


## 5. Local Experiment Runner

This cell turns model, prompt, and hyperparameter choices into an explicit experiment grid. Each model output is scored with the intrinsic metrics, then saved as both a case-level table and an aggregated summary table.


In [47]:
def build_experiment_grid(
    # model_aliases: Iterable[str] = ("gpt_oss_120b", "nemotron", "gemma"),
    model_aliases: Iterable[str] = ("gpt_oss_120b",),
    prompt_ids: Iterable[str] = ("strict_multimedia_v1", "keyword_control_v1"),
    temperatures: Iterable[float] = (0.2, 0.4),
    reasoning_efforts: Iterable[str | None] = ("medium",),
    max_tokens_values: Iterable[int] = (2048,),
    k_values: Iterable[int] = (DEFAULT_K,),
    output_modes: Iterable[str] = ("json_prompt",),
) -> list[dict[str, Any]]:
    """Expand model/prompt/hyperparameter options into explicit run configurations."""
    grid: list[dict[str, Any]] = []
    for model_alias, prompt_id, temperature, reasoning_effort, max_tokens, k, output_mode in product(
        model_aliases,
        prompt_ids,
        temperatures,
        reasoning_efforts,
        max_tokens_values,
        k_values,
        output_modes,
    ):
        grid.append(
            {
                "model_alias": model_alias,
                "prompt_id": prompt_id,
                "temperature": float(temperature),
                "reasoning_effort": reasoning_effort,
                "max_tokens": int(max_tokens),
                "k": int(k),
                "output_mode": output_mode,
            }
        )
    return grid


def flatten_metrics(metrics: dict[str, Any]) -> dict[str, Any]:
    """Flatten metric payloads so CSV columns stay analysis-friendly."""
    flat: dict[str, Any] = {}
    for key, value in metrics.items():
        if isinstance(value, (int, float)) or value is None:
            flat[f"metric_{key}"] = value
        else:
            flat[f"metric_{key}"] = json.dumps(value, ensure_ascii=False)
    return flat


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate case-level results into a model/prompt comparison table."""
    if df.empty:
        return df
    group_cols = [
        "model_alias",
        "model_name",
        "prompt_id",
        "temperature",
        "reasoning_effort",
        "max_tokens",
        "k",
        "output_mode",
    ]
    metric_cols = [col for col in df.columns if col.startswith("metric_") and pd.api.types.is_numeric_dtype(df[col])]
    agg_map: dict[str, list[str] | str] = {col: ["mean", "std"] for col in metric_cols}
    agg_map["latency_sec"] = ["mean", "std"]
    agg_map["case_id"] = "count"
    summary = df.groupby(group_cols, dropna=False).agg(agg_map).reset_index()
    summary.columns = ["_".join(part for part in col if part) if isinstance(col, tuple) else col for col in summary.columns]
    summary = summary.rename(columns={"case_id_count": "num_cases"})
    if "metric_overall_score_mean" in summary.columns:
        summary = summary.sort_values("metric_overall_score_mean", ascending=False)
    return summary


def run_local_experiment(
    grid: list[dict[str, Any]],
    cases: list[dict[str, Any]] = SYNTHETIC_CASES,
    repetitions: int = 1,
    use_embeddings: bool = False,
    experiment_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, Path, Path]:
    """Run the local H1 experiment grid and persist detailed/summary tables."""
    run_id = experiment_name or datetime.now(timezone.utc).strftime("h1_qe_%Y%m%d_%H%M%S")
    prediction_path = EXPERIMENT_DIR / f"{run_id}_predictions.jsonl"
    rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []

    for config_row in grid:
        # Skip unavailable providers instead of failing the whole sweep.
        model_config = MODEL_CONFIGS[config_row["model_alias"]]
        ready, reason = model_is_ready(model_config)
        if not ready:
            print(f"Skipping {model_config.alias}: {reason}")
            continue

        params = GenerationParams(
            temperature=config_row["temperature"],
            max_tokens=config_row["max_tokens"],
            reasoning_effort=config_row["reasoning_effort"],
            output_mode=config_row["output_mode"],
        )

        for case in cases:
            for repetition in range(repetitions):
                base_row = {
                    "run_id": run_id,
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "case_id": case["case_id"],
                    "query_type": case["query_type"],
                    "model_alias": model_config.alias,
                    "model_provider": model_config.provider,
                    "model_name": model_config.model_name,
                    "prompt_id": config_row["prompt_id"],
                    "temperature": params.temperature,
                    "reasoning_effort": params.reasoning_effort,
                    "max_tokens": params.max_tokens,
                    "k": config_row["k"],
                    "output_mode": params.output_mode,
                    "repetition": repetition,
                }
                try:
                    expansion = expand_query(
                        query=case["query"],
                        k=config_row["k"],
                        model_config=model_config,
                        prompt_id=config_row["prompt_id"],
                        params=params,
                    )
                    metrics = compute_query_expansion_metrics(
                        expansion["expanded_queries"],
                        case=case,
                        k=config_row["k"],
                        use_embeddings=use_embeddings,
                    )
                    row = {
                        **base_row,
                        "original_query": case["query"],
                        "expanded_queries_json": json.dumps(expansion["expanded_queries"], ensure_ascii=False),
                        "attempts": expansion["attempts"],
                        "latency_sec": expansion["latency_sec"],
                        "error": None,
                        **flatten_metrics(metrics),
                    }
                    rows.append(row)
                    prediction_rows.append({**row, "usage_events": expansion.get("usage_events", [])})
                except Exception as exc:
                    row = {
                        **base_row,
                        "original_query": case["query"],
                        "expanded_queries_json": "[]",
                        "attempts": 0,
                        "latency_sec": None,
                        "error": repr(exc),
                    }
                    rows.append(row)
                    prediction_rows.append(row)
                    print(f"Error on {model_config.alias}/{config_row['prompt_id']}/{case['case_id']}: {exc}")

    result_df = pd.DataFrame(rows)
    summary_df = summarize_results(result_df[result_df["error"].isna()].copy()) if not result_df.empty else pd.DataFrame()

    write_jsonl(prediction_rows, prediction_path)
    result_path = EXPERIMENT_DIR / f"{run_id}_results.csv"
    summary_path = EXPERIMENT_DIR / f"{run_id}_summary.csv"
    result_df.to_csv(result_path, index=False, encoding="utf-8-sig")
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("Saved predictions:", prediction_path)
    print("Saved case-level table:", result_path)
    print("Saved comparison table:", summary_path)
    return result_df, summary_df, result_path, summary_path


## 6. Run Local Experiments

This cell is disabled by default to avoid accidental API calls. Set `RUN_LOCAL_EXPERIMENT = True`, adjust `GRID`, and run the cell when you are ready to spend model calls.


In [48]:
# Keep the default run small to control cost. Flip RUN_LOCAL_EXPERIMENT to True when ready.
RUN_LOCAL_EXPERIMENT = True

if RUN_LOCAL_EXPERIMENT:
    GRID = build_experiment_grid(
        # model_aliases=("gpt_oss_120b", "gemma", "nemotron"),
        model_aliases=("gpt_oss_120b",),
        prompt_ids=("strict_multimedia_v1", "keyword_control_v1"),
        temperatures=(0.2, 0.4),
        reasoning_efforts=("medium",),
        k_values=(5,),
        output_modes=("json_prompt",),
    )
    result_df, summary_df, result_path, summary_path = run_local_experiment(
        grid=GRID,
        cases=SYNTHETIC_CASES,
        repetitions=2,
        use_embeddings=False,
    )
    display(summary_df.head(20))
else:
    print("Set RUN_LOCAL_EXPERIMENT=True to run model calls.")
    print("Configured model readiness:")
    for alias, config in MODEL_CONFIGS.items():
        print(f"- {alias}: {model_is_ready(config)} | model={config.model_name}")


Saved predictions: D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1\h1_qe_20260626_110945_predictions.jsonl
Saved case-level table: D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1\h1_qe_20260626_110945_results.csv
Saved comparison table: D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1\h1_qe_20260626_110945_summary.csv


,model_alias,model_name,prompt_id,temperature,reasoning_effort,max_tokens,k,output_mode,metric_exact_k_mean,metric_exact_k_std,...,metric_englishish_score_std,metric_length_score_mean,metric_length_score_std,metric_overall_score_mean,metric_overall_score_std,metric_query_drift_risk_mean,metric_query_drift_risk_std,latency_sec_mean,latency_sec_std,num_cases
1,gpt_oss_120b,openai/gpt-oss-120b,keyword_control_v1,0.4,medium,2048,5,json_prompt,1.0,0.0,...,0.007089,0.983333,0.031946,0.861628,0.036944,0.065625,0.076308,7.135383,6.079799,16
0,gpt_oss_120b,openai/gpt-oss-120b,keyword_control_v1,0.2,medium,2048,5,json_prompt,1.0,0.0,...,0.007022,0.988472,0.021225,0.851952,0.033413,0.079687,0.083775,4.860843,3.766132,16
2,gpt_oss_120b,openai/gpt-oss-120b,strict_multimedia_v1,0.2,medium,2048,5,json_prompt,1.0,0.0,...,0.008809,0.947222,0.058443,0.828097,0.039979,0.098437,0.095511,5.717981,5.446916,16
3,gpt_oss_120b,openai/gpt-oss-120b,strict_multimedia_v1,0.4,medium,2048,5,json_prompt,1.0,0.0,...,0.018190,0.931528,0.064683,0.821493,0.050319,0.112500,0.106849,5.304687,4.512574,16


## 7. LangSmith Evaluation Overview

LangSmith is optional for H1. Use it when you want trace-level visibility into model calls and an experiment table that compares custom evaluators across model, prompt, and hyperparameter settings.

Required environment variables:

```text
LANGSMITH_API_KEY=...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=multimodal-retrieval-h1-expand-query
```

The default dataset name is `h1-query-expansion-synthetic-v1`.


## 8. LangSmith Dataset, Target, And Evaluators

This cell syncs the synthetic cases to LangSmith, wraps the local expander as a LangSmith target function, and maps the local H1 metrics into LangSmith evaluator functions.


In [55]:
LANGSMITH_DATASET_NAME = os.getenv("LANGSMITH_H1_DATASET", "h1-query-expansion-synthetic-v1")


def require_langsmith_client() -> Any:
    """Create a LangSmith client after validating optional dependency and API key."""
    if Client is None:
        raise RuntimeError("Install langsmith to use LangSmith evaluation.")
    if not os.getenv("LANGSMITH_API_KEY"):
        raise RuntimeError("Set LANGSMITH_API_KEY before running LangSmith evaluation.")
    return Client()


def sync_langsmith_dataset(cases: list[dict[str, Any]] = SYNTHETIC_CASES, k: int = DEFAULT_K, dataset_name: str = LANGSMITH_DATASET_NAME):
    """Create or update the LangSmith dataset used for H1 query expansion eval."""
    client = require_langsmith_client()
    try:
        dataset = client.read_dataset(dataset_name=dataset_name)
    except Exception:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="Synthetic intrinsic evaluation set for H1 query expansion. Retrieval is intentionally excluded.",
        )

    existing_case_ids: set[str] = set()
    try:
        for example in client.list_examples(dataset_id=dataset.id):
            case_id = getattr(example, "inputs", {}).get("case_id")
            if case_id:
                existing_case_ids.add(case_id)
    except Exception:
        existing_case_ids = set()

    examples = []
    for case in cases:
        if case["case_id"] in existing_case_ids:
            continue
        examples.append(
            {
                "inputs": {
                    "case_id": case["case_id"],
                    "query": case["query"],
                    "query_type": case["query_type"],
                    "k": k,
                },
                "outputs": {
                    "required_concepts": case["required_concepts"],
                    "forbidden_terms": case["forbidden_terms"],
                    "reference_expansions": case["reference_expansions"],
                },
                "metadata": {"phase": "h1", "task": "query_expansion", "retrieval": False},
            }
        )

    if examples:
        client.create_examples(dataset_id=dataset.id, examples=examples)
        print(f"Added {len(examples)} new examples to {dataset_name}")
    else:
        print(f"Dataset {dataset_name} already has these case IDs.")
    return dataset


def case_from_langsmith(inputs: dict[str, Any], reference_outputs: dict[str, Any]) -> dict[str, Any]:
    """Reconstruct an H1 synthetic case from LangSmith example payloads."""
    return {
        "case_id": inputs["case_id"],
        "query_type": inputs.get("query_type", "unknown"),
        "query": inputs["query"],
        "required_concepts": reference_outputs["required_concepts"],
        "forbidden_terms": reference_outputs["forbidden_terms"],
        "reference_expansions": reference_outputs["reference_expansions"],
    }


def _langsmith_metrics(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> dict[str, Any]:
    """Shared metric adapter used by individual LangSmith evaluator functions."""
    case = case_from_langsmith(inputs, reference_outputs)
    expanded_queries = outputs.get("expanded_queries", [])
    return compute_query_expansion_metrics(expanded_queries, case=case, k=inputs.get("k", DEFAULT_K), use_embeddings=False)


def ls_overall_score(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for the weighted H1 score."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["overall_score"])


def ls_concept_coverage(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for required concept coverage."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["required_concept_coverage"])


def ls_forbidden_avoidance(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for hallucination/drift avoidance."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["forbidden_avoidance"])


def ls_diversity(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for lexical diversity."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["pairwise_lexical_diversity"])


def ls_exact_k(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for exact-k compliance."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["exact_k"])


LANGSMITH_EVALUATORS = [
    ls_overall_score,
    ls_concept_coverage,
    ls_forbidden_avoidance,
    ls_diversity,
    ls_exact_k,
]


def make_langsmith_target(model_alias: str, prompt_id: str, params: GenerationParams):
    """Wrap the local expander as a LangSmith target function."""
    model_config = MODEL_CONFIGS[model_alias]

    def _target(inputs: dict[str, Any]) -> dict[str, Any]:
        expansion = expand_query(
            query=inputs["query"],
            k=inputs.get("k", DEFAULT_K),
            model_config=model_config,
            prompt_id=prompt_id,
            params=params,
        )
        return {
            "expanded_queries": expansion["expanded_queries"],
            "attempts": expansion["attempts"],
            "latency_sec": expansion["latency_sec"],
            "model_alias": model_alias,
            "model_name": model_config.model_name,
            "prompt_id": prompt_id,
            "generation_params": asdict(params),
        }

    if traceable is None:
        return _target
    return traceable(name=f"h1_query_expansion/{model_alias}/{prompt_id}")(_target)


def run_langsmith_experiment(
    model_alias: str = "gpt_oss_120b",
    prompt_id: str = "strict_multimedia_v1",
    params: GenerationParams = GenerationParams(),
    dataset_name: str = LANGSMITH_DATASET_NAME,
    max_concurrency: int = 1,
):
    """Run one LangSmith experiment for a selected model/prompt configuration."""
    client = require_langsmith_client()
    dataset = sync_langsmith_dataset(k=DEFAULT_K, dataset_name=dataset_name)
    model_config = MODEL_CONFIGS[model_alias]
    ready, reason = model_is_ready(model_config)
    if not ready:
        raise RuntimeError(f"Model {model_alias} is not ready: {reason}")

    target = make_langsmith_target(model_alias, prompt_id, params)
    metadata = {
        "models": [model_config.model_name],
        "prompts": [prompt_id],
        "phase": "h1",
        "task": "query_expansion",
        "retrieval": False,
        "model_alias": model_alias,
        "provider": model_config.provider,
        "temperature": params.temperature,
        "max_tokens": params.max_tokens,
        "reasoning_effort": params.reasoning_effort,
        "output_mode": params.output_mode,
    }
    return client.evaluate(
        target,
        data=dataset.name,
        evaluators=LANGSMITH_EVALUATORS,
        experiment_prefix=f"h1-qe {model_alias} {prompt_id} t={params.temperature}",
        description="Intrinsic query expansion evaluation. No retrieval metrics are used.",
        max_concurrency=max_concurrency,
        metadata=metadata,
    )


## 9. Run LangSmith Experiment

This cell is also disabled by default. Set `RUN_LANGSMITH_EXPERIMENT = True` only after `LANGSMITH_API_KEY` and the required model provider keys are configured.


In [62]:
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

False

In [66]:
# Example LangSmith run. Keep disabled until LANGSMITH_API_KEY and model API keys are ready.
RUN_LANGSMITH_EXPERIMENT = True

if RUN_LANGSMITH_EXPERIMENT:
    ls_results = run_langsmith_experiment(
        model_alias="gpt_oss_120b",
        prompt_id="strict_multimedia_v1",
        params=GenerationParams(temperature=0.2, reasoning_effort="medium", output_mode="json_prompt"),
        max_concurrency=1,
    )
    print(ls_results)
else:
    print("Set RUN_LANGSMITH_EXPERIMENT=True after configuring LANGSMITH_API_KEY and model keys.")


Added 8 new examples to h1-query-expansion-synthetic-v1
View the evaluation results for experiment: 'h1-qe gpt_oss_120b strict_multimedia_v1 t=0.2-e2bb59db' at:
https://smith.langchain.com/o/f70412f7-4c6b-4c3e-8ca7-23200eb8452d/datasets/84fe34eb-8f1f-4ffe-9226-b5fd146c0e98/compare?selectedSessions=1f45441b-abcb-4a69-ae69-bd6f4197bdfb




8it [00:45,  5.69s/it]

<ExperimentResults h1-qe gpt_oss_120b strict_multimedia_v1 t=0.2-e2bb59db>


## 10. Result Inspection Helpers

These helper functions list generated experiment artifacts, load the newest summary CSV, and display the most useful comparison columns for model and prompt selection.


In [ ]:
def list_experiment_files() -> pd.DataFrame:
    """List generated H1 experiment artifacts in the output directory."""
    rows = []
    for path in sorted(EXPERIMENT_DIR.glob("h1_qe_*")):
        rows.append(
            {
                "name": path.name,
                "path": str(path),
                "size_kb": round(path.stat().st_size / 1024, 2),
                "modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
            }
        )
    return pd.DataFrame(rows)


def load_latest_summary() -> pd.DataFrame:
    """Load the newest summary CSV for quick notebook analysis."""
    summaries = sorted(EXPERIMENT_DIR.glob("h1_qe_*_summary.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not summaries:
        raise FileNotFoundError(f"No summary CSV found in {EXPERIMENT_DIR}")
    return pd.read_csv(summaries[0])


def compare_by_prompt_or_model(summary_df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Return the most useful comparison columns sorted by overall H1 score."""
    score_col = "metric_overall_score_mean"
    display_cols = [
        "model_alias",
        "model_name",
        "prompt_id",
        "temperature",
        "reasoning_effort",
        "output_mode",
        "num_cases",
        score_col,
        "metric_required_concept_coverage_mean",
        "metric_forbidden_avoidance_mean",
        "metric_pairwise_lexical_diversity_mean",
        "latency_sec_mean",
    ]
    available_cols = [col for col in display_cols if col in summary_df.columns]
    return summary_df.sort_values(score_col, ascending=False)[available_cols].head(top_n)


list_experiment_files()


## 11. How To Extend This Pipeline

Recommended H1 extension workflow:

1. Add new cases to `SYNTHETIC_CASES` when you discover new failure modes.
2. Add new prompt variants to `PROMPT_VARIANTS` with clear versioned names.
3. Add new providers or model IDs to `MODEL_CONFIGS`.
4. Expand `build_experiment_grid(...)` to test model, prompt, temperature, `k`, reasoning effort, max tokens, output mode, and repetitions.
5. Compare local `*_summary.csv` files, then inspect detailed predictions in JSONL or LangSmith traces.

When the retrieval pipeline is ready, use the best H1 configurations as candidates for H2 retrieval experiments and then add Recall@K, MRR, nDCG, and related corpus-based metrics.
